# Compress an MP4 below 25 MiB for GitHub
This notebook uploads a screen recording, removes a silent audio track, and compresses the video to a safe target below GitHub's browser-upload limit. The default settings match the AeroChef demo: 960×540, 15 fps, H.264.


In [ ]:
from google.colab import files
uploaded = files.upload()
INPUT = next(iter(uploaded))
OUTPUT = 'aerochef_demo_compressed.mp4'
TARGET_MIB = 24.0
WIDTH = 960
FPS = 15
print('Input:', INPUT)


In [ ]:
import json, subprocess, os
probe = subprocess.run([
    'ffprobe','-v','error','-show_entries','format=duration',
    '-of','json', INPUT
], check=True, capture_output=True, text=True)
duration = float(json.loads(probe.stdout)['format']['duration'])
target_bits = TARGET_MIB * 1024 * 1024 * 8
video_kbps = max(150, int((target_bits / duration / 1000) * 0.92))
print(f'Duration: {duration:.2f} seconds')
print(f'Target video bitrate: {video_kbps} kbps')


In [ ]:
command = [
    'ffmpeg','-y','-hide_banner','-loglevel','error',
    '-i', INPUT,
    '-an',  # The AeroChef source audio is silent; remove it.
    '-vf', f'fps={FPS},scale={WIDTH}:-2:flags=bilinear',
    '-c:v','libx264','-preset','ultrafast','-tune','zerolatency',
    '-b:v',f'{video_kbps}k',
    '-maxrate',f'{int(video_kbps*1.18)}k',
    '-bufsize',f'{int(video_kbps*2.36)}k',
    '-profile:v','baseline','-pix_fmt','yuv420p',
    '-movflags','+faststart', OUTPUT
]
subprocess.run(command, check=True)
size_mib = os.path.getsize(OUTPUT) / 1024 / 1024
print(f'Created {OUTPUT}: {size_mib:.2f} MiB')
assert size_mib < 25, 'Output is still too large; reduce TARGET_MIB and rerun.'


In [ ]:
files.download(OUTPUT)
